# Tutorial 0: Step Control with Batch JVP $\rho$

This notebook is the intended first stop for a new reader.

What it shows:
- how to estimate a batch-level convergence radius $\rho$ with one JVP,
- how a plain SGD step changes when we introduce a $\rho$ cap,
- how a fully geometry-set update differs from a fixed learning rate,
- and why the difference is about **step length**, not about changing the gradient direction.

We use `sklearn` digits as a lightweight stand-in for MNIST so the notebook stays fast and interactive.


In [ ]:
from pathlib import Path
import importlib.util
import sys


def find_repo_root():
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        if (base / "src" / "ghosts").exists() and (base / "tutorials").exists():
            return base
    raise RuntimeError("Run this notebook from inside the ghosts-of-softmax repository.")


REPO = find_repo_root()
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"Repo root: {REPO}")


## 1. Setup: data, runtime, and one-switch configuration

The default path is deliberately simple:
- one architecture,
- one seed,
- two learning rates,
- three update rules.

After you understand the single-seed story, you can flip on the optional multiseed block at the end.


In [ ]:
import math
import random
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch.func import functional_call, jvp
from torch.utils.data import DataLoader, TensorDataset

torch.set_num_threads(1)

DEVICE = torch.device("cpu")
ARCH_NAME = "MLP"
LRS = [0.03, 0.20]
EPOCHS = 14
BATCH_SIZE = 128
TARGET_R = 1.0
SEED = 7

RUN_MULTI_SEED = False
MULTI_SEEDS = [0, 1, 2, 3, 4]

PALETTE = {
    "sgd": "#C03A2B",
    "rho_capped": "#1F77B4",
    "rho_set": "#FF7F0E",
    "gray": "#BDBDBD",
    "dark": "#3D3D3D",
}

LABELS = {
    "sgd": "Fixed SGD",
    "rho_capped": "rho-capped SGD",
    "rho_set": "rho-set SGD",
}


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def build_digits_loaders(seed: int, batch_size: int = 128):
    digits = load_digits()
    X = digits.data.astype(np.float32) / 16.0
    y = digits.target.astype(np.int64)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=seed, stratify=y
    )
    train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
    test_ds = TensorDataset(torch.tensor(X_test), torch.tensor(y_test))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)
    return train_loader, test_loader


train_loader, test_loader = build_digits_loaders(SEED, BATCH_SIZE)
len(train_loader), len(test_loader)

## 2. Small architectures

We start with an `MLP`, because it is easy to reason about and trains quickly.

The notebook also defines a linear model and a small CNN so you can reuse the same controller idea later. The important point is that **the controller logic does not depend on the architecture**. It only needs:
- a proposed update direction,
- and a batch-level estimate of $\rho$ along that direction.


In [ ]:
class LinearModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(64, 10)

    def forward(self, x):
        return self.fc(x)


class MLP(nn.Module):
    def __init__(self, width=128):
        super().__init__()
        self.fc1 = nn.Linear(64, width)
        self.fc2 = nn.Linear(width, width)
        self.fc3 = nn.Linear(width, 10)

    def forward(self, x):
        x = F.gelu(self.fc1(x))
        x = F.gelu(self.fc2(x))
        return self.fc3(x)


class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.fc1 = nn.Linear(32 * 2 * 2, 64)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        x = x.view(-1, 1, 8, 8)
        x = F.gelu(self.conv1(x))
        x = F.max_pool2d(x, 2)
        x = F.gelu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = x.flatten(1)
        x = F.gelu(self.fc1(x))
        return self.fc2(x)


ARCHS = {
    "Linear": LinearModel,
    "MLP": MLP,
    "CNN": SmallCNN,
}


def make_model(name: str):
    return ARCHS[name]().to(DEVICE)


make_model(ARCH_NAME)


## 3. Batch JVP estimate of $\rho$

This tutorial uses a batch-level JVP estimate.

For the current batch, we:
1. compute the gradient direction,
2. push that direction through the logits with a JVP,
3. measure the largest directional logit spread,
4. convert it to a radius estimate with $\rho = \pi / \Delta_a$.

The key idea is that **plain SGD, rho-capped SGD, and rho-set SGD all start from the same gradient direction**. The controller only changes the step length.


In [ ]:
@dataclass
class EpochStats:
    train_loss: float
    test_acc: float
    mean_r: float
    max_r: float
    mean_rho: float
    mean_tau: float
    mean_eta: float
    cap_fraction: float


def evaluate(model: nn.Module, loader: DataLoader):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            logits = model(xb)
            total_loss += F.cross_entropy(logits, yb, reduction="sum").item()
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += len(yb)
    return total_loss / total, correct / total


def grad_dict(model: nn.Module):
    grads = {}
    sq = 0.0
    for name, param in model.named_parameters():
        if param.grad is None:
            continue
        g = param.grad.detach().clone()
        grads[name] = g
        sq += float((g * g).sum().item())
    norm = math.sqrt(max(sq, 1e-12))
    return grads, norm


def unit_direction_from_grads(grads):
    sq = sum(float((g * g).sum().item()) for g in grads.values())
    norm = math.sqrt(max(sq, 1e-12))
    return {name: -g / norm for name, g in grads.items()}, norm


def batch_rho_jvp(model: nn.Module, xb: torch.Tensor, direction):
    params = {name: param.detach() for name, param in model.named_parameters()}

    def f(pdict):
        return functional_call(model, pdict, (xb,))

    logits = f(params).detach()
    _, dlogits = jvp(f, (params,), (direction,))
    spread = dlogits.max(dim=1).values - dlogits.min(dim=1).values
    delta_a = float(spread.max().item())
    rho = math.pi / max(delta_a, 1e-12)
    return rho, delta_a, logits


def apply_sgd_update(model: nn.Module, eta_eff: float):
    with torch.no_grad():
        for param in model.parameters():
            if param.grad is not None:
                param.add_(param.grad, alpha=-eta_eff)


def run_training(mode: str, base_lr: float, seed: int, arch_name: str, target_r: float = 1.0):
    assert mode in {"sgd", "rho_capped", "rho_set"}
    set_seed(seed)
    train_loader, test_loader = build_digits_loaders(seed, BATCH_SIZE)
    model = make_model(arch_name)

    history = {key: [] for key in [
        "train_loss", "test_acc", "mean_r", "max_r", "mean_rho",
        "mean_tau", "mean_eta", "cap_fraction"
    ]}

    for epoch in range(EPOCHS):
        model.train()
        batch_losses = []
        batch_r = []
        batch_rho = []
        batch_tau = []
        batch_eta = []
        batch_capped = []

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            model.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = F.cross_entropy(logits, yb)
            loss.backward()

            grads, grad_norm = grad_dict(model)
            direction, grad_norm = unit_direction_from_grads(grads)
            rho_batch, delta_a, _ = batch_rho_jvp(model, xb, direction)

            tau_raw = base_lr * grad_norm
            tau_cap = target_r * rho_batch

            if mode == "sgd":
                tau = tau_raw
                eta_eff = base_lr
                capped = 0.0
            elif mode == "rho_capped":
                tau = min(tau_raw, tau_cap)
                eta_eff = tau / max(grad_norm, 1e-12)
                capped = float(tau_raw > tau_cap)
            else:
                tau = tau_cap
                eta_eff = tau / max(grad_norm, 1e-12)
                capped = 1.0

            r = tau / max(rho_batch, 1e-12)
            apply_sgd_update(model, eta_eff)

            batch_losses.append(float(loss.item()))
            batch_r.append(r)
            batch_rho.append(rho_batch)
            batch_tau.append(tau)
            batch_eta.append(eta_eff)
            batch_capped.append(capped)

        test_loss, test_acc = evaluate(model, test_loader)
        history["train_loss"].append(float(np.mean(batch_losses)))
        history["test_acc"].append(float(test_acc))
        history["mean_r"].append(float(np.mean(batch_r)))
        history["max_r"].append(float(np.max(batch_r)))
        history["mean_rho"].append(float(np.mean(batch_rho)))
        history["mean_tau"].append(float(np.mean(batch_tau)))
        history["mean_eta"].append(float(np.mean(batch_eta)))
        history["cap_fraction"].append(float(np.mean(batch_capped)))

    return history


def final_summary_row(mode_name, lr, hist):
    return {
        "mode": mode_name,
        "base_lr": lr,
        "final_acc": hist["test_acc"][-1],
        "peak_r": max(hist["max_r"]),
        "final_mean_eta": hist["mean_eta"][-1],
        "final_cap_fraction": hist["cap_fraction"][-1],
    }


## 4. Single-seed run: same architecture, same data, same base learning rates

Here is the central comparison.

For each base learning rate, we run three update rules:
- **fixed SGD**: always use the same learning rate,
- **rho-capped SGD**: start from the same SGD proposal, but shorten the step if it would exceed $\rho$,
- **rho-set SGD**: ignore the base learning rate and set the step length directly from geometry, $\tau = r_{\text{target}}\rho$.

The only thing that changes between fixed SGD and rho-capped SGD is the **step length** on batches where the raw SGD proposal is too aggressive.


In [ ]:
single_seed = {}
for lr in LRS:
    single_seed[lr] = {
        "sgd": run_training("sgd", lr, SEED, ARCH_NAME, target_r=TARGET_R),
        "rho_capped": run_training("rho_capped", lr, SEED, ARCH_NAME, target_r=TARGET_R),
        "rho_set": run_training("rho_set", lr, SEED, ARCH_NAME, target_r=TARGET_R),
    }

summary_rows = []
for lr in LRS:
    for mode in ["sgd", "rho_capped", "rho_set"]:
        summary_rows.append(final_summary_row(mode, lr, single_seed[lr][mode]))
summary_rows


## 5. What changed when we turned on the controller?

Start with the two outputs that are easiest to read:
- **training loss** on a log scale,
- **test accuracy** on a linear scale.

The question is simple: after we add the controller, do we get a visibly different optimization trajectory?

Read the figure with this checklist in mind:
1. **Color = optimizer mode**: red is fixed SGD, blue is rho-capped SGD, orange is rho-set SGD.
2. **Row = base learning rate**: the top row uses a conservative LR, the bottom row uses an aggressive LR. This lets you see when the controller matters.
3. **Loss**: does the aggressive fixed-LR run become unstable while the controlled run stays smoother?
4. **Accuracy**: does rho-capped SGD preserve accuracy when plain SGD starts to degrade?

We leave the detailed controller diagnostics ($r$, effective learning rate, cap frequency) for later. First, focus on the visible change in optimization behavior.

In [ ]:
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica Neue", "DejaVu Sans"],
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

epochs = np.arange(1, EPOCHS + 1)
marker_map = {"sgd": "o", "rho_capped": "s", "rho_set": "^"}
modes = ["sgd", "rho_capped", "rho_set"]

fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True)
fig.suptitle(
    "Capping step length to the convergence radius prevents loss spikes",
    fontsize=12, fontweight="bold",
)

for row, lr in enumerate(LRS):
    runs = single_seed[lr]
    ax_loss = axes[row, 0]
    ax_acc = axes[row, 1]

    for mode in modes:
        color = PALETTE[mode]
        marker = marker_map[mode]
        label = LABELS[mode]
        data_loss = runs[mode]["train_loss"]
        data_acc = runs[mode]["test_acc"]

        ax_loss.semilogy(
            epochs, data_loss,
            color=color, lw=2.2, marker=marker,
            ms=4.5, markevery=(1, 2), label=label,
        )
        ax_acc.plot(
            epochs, data_acc,
            color=color, lw=2.2, marker=marker,
            ms=4.5, markevery=(1, 2), label=label,
        )

    ax_acc.set_ylim(0.0, 1.02)
    for ax in [ax_loss, ax_acc]:
        ax.yaxis.grid(True, alpha=0.2, lw=0.5)
        ax.text(
            0.02, 0.95, f"LR = {lr}",
            transform=ax.transAxes, fontsize=10,
            fontweight="bold", va="top", color=PALETTE["dark"],
        )

    if row == 1:
        ax_loss.set_xlabel("epoch")
        ax_acc.set_xlabel("epoch")

axes[0, 0].set_title("Training loss")
axes[0, 1].set_title("Test accuracy")
axes[0, 1].legend(frameon=False, fontsize=9, loc="lower right")

fig.tight_layout()
plt.show()

## 6. A compact summary table

This table makes the tutorial's main point visible in one place.

When rho-capped SGD differs from fixed SGD, it is not because it discovered a different descent direction. It is because the controller noticed that the proposed step was too long relative to the local radius and shortened it.


In [ ]:
summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df[[
    "mode", "base_lr", "final_acc", "peak_r", "final_mean_eta", "final_cap_fraction"
]].sort_values(["base_lr", "mode"]).reset_index(drop=True)
summary_df = summary_df.round({
    "base_lr": 2,
    "final_acc": 3,
    "peak_r": 3,
    "final_mean_eta": 4,
    "final_cap_fraction": 3,
})
summary_df


## 7. Optional multiseed check

Once the single-seed behavior is clear, you can turn on `RUN_MULTI_SEED` and rerun this section.

This is intentionally optional. A tutorial should first help you understand the mechanism, and only then pay the extra cost of reproducibility summaries.


In [ ]:
if RUN_MULTI_SEED:
    multi_rows = []
    for lr in LRS:
        per_seed = {"sgd": [], "rho_capped": [], "rho_set": []}
        for seed in MULTI_SEEDS:
            for mode in per_seed:
                hist = run_training(mode, lr, seed, ARCH_NAME, target_r=TARGET_R)
                per_seed[mode].append(final_summary_row(mode, lr, hist))

        for mode in ["sgd", "rho_capped", "rho_set"]:
            accs = [row["final_acc"] for row in per_seed[mode]]
            peak_rs = [row["peak_r"] for row in per_seed[mode]]
            caps = [row["final_cap_fraction"] for row in per_seed[mode]]
            multi_rows.append({
                "mode": mode,
                "base_lr": lr,
                "acc_median": float(np.median(accs)),
                "acc_iqr": float(np.percentile(accs, 75) - np.percentile(accs, 25)),
                "peak_r_median": float(np.median(peak_rs)),
                "cap_fraction_median": float(np.median(caps)),
            })

    for row in multi_rows:
        print(
            f"mode={row['mode']:<10} base_lr={row['base_lr']:<4.2f} "
            f"acc_median={row['acc_median']:.3f} acc_iqr={row['acc_iqr']:.3f} "
            f"peak_r_median={row['peak_r_median']:.3f} "
            f"cap_fraction_median={row['cap_fraction_median']:.3f}"
        )
else:
    print("RUN_MULTI_SEED is False. Flip it to True if you want the slower multiseed summary.")


## Where to go next

After this notebook, the intended sequence is:
1. `01_binary_radius.ipynb` for the exact binary formula,
2. `03_rho_controller.ipynb` for the packaged controller API,
3. figure notebooks only after the controller idea is already familiar.
